# Best Practices for Effective Prompts

Source slide: `slides/prompt-engineering/06_Best_Practices_for_Effective_Prompts.md`

These examples map directly to the best-practice techniques named in the slide.



## Prerequisites

- Install the real GitHub Copilot SDK: `pip install github-copilot-sdk`
- Sign in to GitHub Copilot in VS Code or GitHub Codespaces
- These examples use ambient auth; do not paste a PAT into the notebook

Each technique below has a real Copilot SDK failure run, a short failure test, and an improved run that shows the fix.


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler

model = "gpt-4.1"

async def ask_copilot(
    prompt: str,
    *,
    system_message: str | None = None,
    model_name: str = model,
) -> str:
    """Run a single prompt through the real GitHub Copilot SDK using ambient auth."""
    async with CopilotClient() as client:
        session_kwargs = {
            "model": model_name,
            "on_permission_request": PermissionHandler.approve_all,
        }
        if system_message:
            session_kwargs["system_message"] = {
                "mode": "append",
                "content": system_message,
            }
        async with await client.create_session(**session_kwargs) as session:
            response = await session.send_and_wait(prompt)
            return response.data.content if response and response.data else ""

def show_block(title: str, content: str) -> None:
    print(title)
    print("=" * 80)
    print(content.strip())
    print()

async def run_prompt_pair(
    *,
    technique_name: str,
    failure_prompt: str,
    improved_prompt: str,
    failure_test: str,
    fix_explanation: str,
    system_message: str | None = None,
    config_note: str | None = None,
) -> None:
    show_block("📘 Technique", technique_name)
    if config_note:
        show_block("⚙️ Configuration note", config_note)
    show_block("❌ Failure prompt", failure_prompt)
    failure_result = await ask_copilot(
        failure_prompt,
        system_message=system_message,
    )
    show_block("❌ Failure result", failure_result)
    show_block("🧪 Failure test", failure_test)
    show_block("✅ Improved prompt", improved_prompt)
    improved_result = await ask_copilot(
        improved_prompt,
        system_message=system_message,
    )
    show_block("✅ Improved result", improved_result)
    show_block("🔧 Why this fixes it", fix_explanation)

print("✅ Setup complete - real GitHub Copilot SDK with ambient auth")
print(f"📦 Using model: {model}")


## Define Clear Objectives

**Failure mode:** If the goal is unclear, the model has to guess what success looks like.

**Failure test:** The failure prompt asks for help with a report but never states the role, audience, or deliverable.

**Technique:** Name the exact goal, role, and output you want.

**Improved example:** The improved prompt defines the report type and target audience, which gives the model a concrete job to do.


In [ ]:
await run_prompt_pair(
    technique_name='Define Clear Objectives',
    failure_prompt='Help with this report.',
    improved_prompt='Write a 5-bullet executive summary for a quarterly platform reliability report aimed at engineering leadership.',
    failure_test='The failure prompt asks for help with a report but never states the role, audience, or deliverable.',
    fix_explanation='The improved prompt defines the report type and target audience, which gives the model a concrete job to do.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Provide Context and Background

**Failure mode:** Context lets the model adapt its answer to the real situation instead of a generic average case.

**Failure test:** The failure prompt asks for rollout advice but provides no information about audience, product tier, or deployment constraints.

**Technique:** Supply the facts the model needs to tailor its answer.

**Improved example:** The improved prompt includes the customer segment and deployment constraints, so the advice can fit the actual scenario.


In [ ]:
await run_prompt_pair(
    technique_name='Provide Context and Background',
    failure_prompt='Recommend how we should roll this feature out.',
    improved_prompt='Recommend how we should roll out this feature for enterprise customers. Context: customers require SSO, changes must be reversible within 30 minutes, and success will be measured by login completion rate.',
    failure_test='The failure prompt asks for rollout advice but provides no information about audience, product tier, or deployment constraints.',
    fix_explanation='The improved prompt includes the customer segment and deployment constraints, so the advice can fit the actual scenario.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Demonstrate with Examples

**Failure mode:** Examples reduce ambiguity about style, labels, and edge cases.

**Failure test:** The failure prompt asks for classification but never shows how the labels should be applied.

**Technique:** Include one or more examples that demonstrate the intended mapping.

**Improved example:** The improved prompt supplies examples, so the model has a pattern to continue instead of inventing its own.


In [ ]:
await run_prompt_pair(
    technique_name='Demonstrate with Examples',
    failure_prompt='Label this message as praise, issue, or request: “Please add CSV export to the dashboard.”',
    improved_prompt='Label each message as praise, issue, or request.\n\nMessage: “The new dashboard is fast.”\nLabel: praise\n\nMessage: “The export button throws an error.”\nLabel: issue\n\nMessage: “Please add CSV export to the dashboard.”\nLabel:',
    failure_test='The failure prompt asks for classification but never shows how the labels should be applied.',
    fix_explanation='The improved prompt supplies examples, so the model has a pattern to continue instead of inventing its own.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Be Precise and Descriptive

**Failure mode:** Precise wording removes room for the model to interpret the task too broadly.

**Failure test:** The failure prompt asks for a summary without stating tone, length, or structure, so the result can drift into marketing or essay mode.

**Technique:** Specify structure, tone, scope, and length directly.

**Improved example:** The improved prompt tells the model exactly how many bullets to produce and what tone to use.


In [ ]:
await run_prompt_pair(
    technique_name='Be Precise and Descriptive',
    failure_prompt='Summarize this release.',
    improved_prompt='Summarize this release in exactly three factual bullets for internal engineers. Avoid marketing language.',
    failure_test='The failure prompt asks for a summary without stating tone, length, or structure, so the result can drift into marketing or essay mode.',
    fix_explanation='The improved prompt tells the model exactly how many bullets to produce and what tone to use.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Iterate and Experiment

**Failure mode:** Prompt engineering improves through observation and revision. Treating the first draft as final leaves avoidable errors in place.

**Failure test:** The failure prompt asks for a final answer immediately, which leaves no room to compare variants or revise weak output.

**Technique:** Generate, inspect, and revise prompts deliberately.

**Improved example:** The improved prompt asks the model to produce a first attempt and then refine it, modeling an iteration loop instead of a one-shot handoff.


In [ ]:
await run_prompt_pair(
    technique_name='Iterate and Experiment',
    failure_prompt='Draft a migration checklist for this API change.',
    improved_prompt='Draft a migration checklist for this API change. Then review your first checklist for missing rollout or rollback steps and provide an improved final version.',
    failure_test='The failure prompt asks for a final answer immediately, which leaves no room to compare variants or revise weak output.',
    fix_explanation='The improved prompt asks the model to produce a first attempt and then refine it, modeling an iteration loop instead of a one-shot handoff.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Emphasize Positive Instructions

**Failure mode:** Negative-only prompting tells the model what to avoid, but it may still lack a clear target behavior.

**Failure test:** The failure prompt says what not to do without specifying the desired answer shape, so the result can still be unfocused.

**Technique:** Lead with what the model should produce, not only what it should avoid.

**Improved example:** The improved prompt gives the desired behavior explicitly, which is easier for the model to follow than a list of prohibitions.


In [ ]:
await run_prompt_pair(
    technique_name='Emphasize Positive Instructions',
    failure_prompt='Do not be vague. Do not be verbose. Do not miss anything when you summarize this incident.',
    improved_prompt='Summarize this incident in exactly three factual bullets. Each bullet must state one cause, impact, or next step.',
    failure_test='The failure prompt says what not to do without specifying the desired answer shape, so the result can still be unfocused.',
    fix_explanation='The improved prompt gives the desired behavior explicitly, which is easier for the model to follow than a list of prohibitions.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Consider Information Order

**Failure mode:** The order of prompt content changes what the model attends to first.

**Failure test:** The failure prompt buries the output rule after the context, so the model may never prioritize it.

**Technique:** Put the main task and critical rules near the top, then follow with supporting context.

**Improved example:** The improved prompt leads with the task and format, so the model sees the most important instructions first.


In [ ]:
await run_prompt_pair(
    technique_name='Consider Information Order',
    failure_prompt='Context: Annual plans save 20%, enterprise customers get SSO, and monthly plans can cancel anytime. After reading all of that, summarize the main benefits in two bullets and include a quote.',
    improved_prompt='Summarize the main benefits in exactly two bullets and include one quoted phrase as evidence.\n\nContext: Annual plans save 20%, enterprise customers get SSO, and monthly plans can cancel anytime.',
    failure_test='The failure prompt buries the output rule after the context, so the model may never prioritize it.',
    fix_explanation='The improved prompt leads with the task and format, so the model sees the most important instructions first.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Provide Alternative Paths

**Failure mode:** Alternative paths prevent the model from fabricating an answer when the preferred path is unavailable.

**Failure test:** The failure prompt assumes the answer exists and gives no fallback, so the model may invent missing facts.

**Technique:** Tell the model what to do when the preferred answer cannot be produced.

**Improved example:** The improved prompt adds an explicit fallback path, which is safer than forcing the model to answer at all costs.


In [ ]:
await run_prompt_pair(
    technique_name='Provide Alternative Paths',
    failure_prompt='Answer the customer’s question about SOC 2 controls.',
    improved_prompt='Answer the customer’s question about SOC 2 controls using the provided context. If the context is missing the answer, reply with “insufficient information” and list the missing document you would need.',
    failure_test='The failure prompt assumes the answer exists and gives no fallback, so the model may invent missing facts.',
    fix_explanation='The improved prompt adds an explicit fallback path, which is safer than forcing the model to answer at all costs.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Optimize Token Usage

**Failure mode:** Long prompts can hide the important instruction and waste context budget.

**Failure test:** The failure prompt uses unnecessary boilerplate and a vague output request, which increases cost without improving clarity.

**Technique:** Trim repeated boilerplate and ask for only the detail you need.

**Improved example:** The improved prompt is shorter, clearer, and tells the model to be concise, which makes the exchange cheaper and easier to follow.


In [ ]:
await run_prompt_pair(
    technique_name='Optimize Token Usage',
    failure_prompt='Please carefully, thoughtfully, and comprehensively explain the following topic with all relevant details you can think of, while also trying to be helpful and complete: onboarding metrics.',
    improved_prompt='Explain onboarding metrics in one short paragraph and then list the top three metrics as bullets.',
    failure_test='The failure prompt uses unnecessary boilerplate and a vague output request, which increases cost without improving clarity.',
    fix_explanation='The improved prompt is shorter, clearer, and tells the model to be concise, which makes the exchange cheaper and easier to follow.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
